# Kryptos-bench — `GLM-4.6-Flash-text`

Runs [`sartajbhuvaji/kryptos-bench`](https://huggingface.co/datasets/sartajbhuvaji/kryptos-bench)
against [`sartajbhuvaji/GLM-4.6-Flash-text`](https://huggingface.co/sartajbhuvaji/GLM-4.6-Flash-text)
served by vLLM on a Lambda Cloud A100, and writes the scored results into
`GLM-4.6-Flash-text/results/`.

## What the benchmark measures

Kryptos-bench is a **contamination-resistant** test of classical cryptanalysis, modelled on
the Kryptos sculpture at CIA headquarters. It has two halves, and the distinction is the
whole point of the design:

| Config | Rows | What it is |
|---|---|---|
| `baseline` | 4 | The real K1–K4 ciphertexts. **Published, and therefore memorisable.** |
| `isomorph_quagmire` | 50 | Fresh Quagmire III instances over literary source text |
| `isomorph_transposition` | 50 | Fresh route/columnar transpositions |
| `isomorph_composite` | 50 | Fresh multi-layer combinations |
| `isomorph_nulls` | 50 | Fresh instances with null characters inserted |

K1–K3 and their keys are all over the internet, so a low error rate on `baseline` shows
**recall**, not cryptanalysis. The four `isomorph_*` configs are generated with the same
cipher machinery but novel keys and novel plaintext, so they cannot have been seen. The
gap between the two is the headline result: it separates a model that has memorised
Kryptos from one that can actually break a cipher.

Every row carries an HLE-style `canary` string so future training-set audits can detect
ingestion.

## Task framing (tiers)

A tier is a *framing over the same row*, not a different dataset. `--tier` overrides it;
by default each row is posed at the tier derived from its own content:

| Tier | Framing | Pass mark |
|---|---|---|
| 1 | Cipher and full key supplied — pure execution | CER = 0.0 |
| 2 | Ciphertext only, substitution family | CER ≤ 0.05 |
| 3 | Ciphertext only, transposition family | CER ≤ 0.10 |
| 4 | K4 — unsolved, no reference plaintext exists | *none* |

These pass marks are **asserted by the design document, not yet calibrated** against an
observed score distribution. This run is one of the inputs to that calibration.

Tier 4 (K4) is scored by crib placement and quadgram fitness rather than CER, because
only 24 plaintext characters of K4 are known and there is nothing to compute an error
rate against.

## How this run is configured

- **Paradigm `cot`.** The `tool_use` paradigm needs a server-side Python sandbox, which
  the OpenAI wire format has no equivalent for. The harness raises rather than silently
  downgrading it, so a local vLLM model gets chain of thought only.
- **Raw ciphertext**, not `--delimited`. One arm per instance.
- **`reasoning_effort` is sent.** vLLM 0.28 genuinely validates this field — a bogus
  value returns HTTP 400 naming the accepted levels, where an unknown field would just
  be ignored with a 200 — so it is passed through and the results record `effort: high`.
  That records what was *sent*, which is the harness's contract; it is not a claim that
  the model does anything with it. In practice this model answers in ~150 tokens
  regardless, so the field appears to be inert here. The cell below re-checks at runtime
  and adds `--no-reasoning-effort` automatically if a future server rejects it.

Prompts are byte-identical to those any other provider would receive — the harness
asserts this in both directions — so numbers here are comparable to a Claude or GPT run
of the same config, tier and paradigm.

---
## 1. Serving the model

The model runs on a Lambda A100 (40 GB, sm80). It is 9.4B params in BF16 — 18.8 GB of
weights — so it fits on one card with ~17 GB left for KV cache. KV is 40 KB/token here
(40 layers × 2 GQA KV heads × 128 head_dim × 2 × 2 bytes).

On the instance:

```bash
export PATH="$HOME/venvs/vllm/bin:$PATH"
export VLLM_USE_FLASHINFER_SAMPLER=0     # see below -- a bare `vllm serve` crashes
~/venvs/vllm/bin/vllm serve sartajbhuvaji/GLM-4.6-Flash-text \
  --served-model-name GLM-4.6-Flash-text \
  --host 0.0.0.0 --port 8000 \
  --max-model-len 32768 \
  --gpu-memory-utilization 0.92 \
  --dtype bfloat16
```

Both exports are load-bearing. vLLM 0.28 pulls in flashinfer, whose sampler JIT-compiles
at the first generated token and dies with `FileNotFoundError: 'ninja'` — pip puts
`ninja` inside the venv, which is not on `PATH` when the binary is exec'd directly.
Fixing only `PATH` moves the failure one step later into an `nvcc` build, where the
system `nvcc` (12.8) mismatches torch's CUDA (13.0). Disabling the flashinfer sampler is
what actually avoids it. This is saved as `~/serve.sh` on the instance — restart with
that, never a bare `vllm serve`.

vLLM reports **453,728 tokens of KV cache (17.31 GiB)** with these flags, which is
13.85 full-length 32k sequences, or ~45 at the ~10k a benchmark instance actually uses.
Measured single-stream decode is **61.8 tok/s** — about 75% of the 82 tok/s memory
bandwidth roof (18.8 GB of weights over 1555 GB/s), so the serving path is healthy.

There is **no reasoning parser configured**. It does not matter here, because under a
strict `json_schema` the grammar constrains output from token 0 and no `<think>` block is
emitted — but an unstructured request against this endpoint would put the think block
into `message.content`.

Then, **from the machine running this notebook**, open a tunnel and leave it open:

```bash
ssh -N -L 8000:localhost:8000 -i ~/.ssh/lambda_glm ubuntu@<INSTANCE_IP>
```

The tunnel is what makes `BASE_URL` below reachable. Lambda's hosted Jupyter is not an
alternative — its proxy rejects programmatic access (XSRF 403s), so drive the endpoint
over SSH rather than through the browser IDE.

In [ ]:
# --- configuration -------------------------------------------------------------------
import os, pathlib, sys

# The vLLM endpoint, reached through the SSH tunnel described above.
BASE_URL = os.environ.get("KRYPTOS_BASE_URL", "http://localhost:8000/v1")
MODEL    = "GLM-4.6-Flash-text"          # must match vLLM's --served-model-name
HF_MODEL = "sartajbhuvaji/GLM-4.6-Flash-text"

# vLLM does not check the key unless started with --api-key, but the harness refuses to
# run with no key at all rather than sending an empty credential.
os.environ.setdefault("VLLM_API_KEY", "EMPTY")
API_KEY_ENV = "VLLM_API_KEY"

CONFIGS = [
    "baseline",
    "isomorph_quagmire",
    "isomorph_transposition",
    "isomorph_composite",
    "isomorph_nulls",
]

# Instances answered in parallel. vLLM batches server-side, so requests in flight become
# one fused forward pass rather than queued serial work -- this is close to free speedup
# until KV cache runs out.
#
# The ceiling is arithmetic: ~17 GB of KV is left after the 18.8 GB of weights, KV costs
# 40 KB/token here, so ~425k tokens fit. A worst-case sequence is ~2k prompt + MAX_TOKENS
# output, so 425000 / (2048 + MAX_TOKENS) ~= 41 sequences at full length. 24 leaves
# headroom for the long tail; past the ceiling vLLM preempts and recomputes, which is
# slower than not having asked. The probe cell below measures where the real knee is.
CONCURRENCY = 24

# The harness defaults to max_tokens=32000, which a 32768-token context window cannot
# grant once the prompt is in it -- vLLM would reject every request. There is no CLI flag
# for it, but --provider-param merges into the raw request last and so wins.
MAX_TOKENS = 8192

REPO    = pathlib.Path.cwd()
while not (REPO / "pyproject.toml").exists() and REPO != REPO.parent:
    REPO = REPO.parent
SRC     = REPO / "src"
RESULTS = REPO / MODEL / "results"
RESULTS.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(SRC))

print(f"repo     {REPO}")
print(f"results  {RESULTS}")
print(f"endpoint {BASE_URL}  ->  {MODEL}")

In [ ]:
# --- dependencies --------------------------------------------------------------------
# `datasets` can take several minutes to install on a Windows-mounted filesystem.
import importlib.util

missing = [p for p in ("openai", "datasets", "rapidfuzz", "matplotlib")
           if importlib.util.find_spec(p) is None]
if missing:
    pkgs = " ".join(missing)          # a plain name: IPython's {...} expansion is happier
    print("installing:", pkgs)
    !{sys.executable} -m pip install -q {pkgs}
else:
    print("all dependencies present")

from kryptos.eval import paradigms, report as reporting, results as results_mod, run_benchmark, tiers
print("harness imported from", pathlib.Path(run_benchmark.__file__).parent)

---
## 2. Endpoint health check

Three things have to be true before spending a run on this endpoint, and each has failed
in practice:

1. the served model name matches what the harness will ask for,
2. **strict `json_schema` structured output works** — the harness sends
   `response_format: {type: "json_schema", json_schema: {strict: true, ...}}` and parses
   the reply as JSON. A server that ignores the schema returns prose, every attempt
   records `unparsed_response`, and the run scores zero for a reason that has nothing to
   do with the model's cryptanalysis,
3. `reasoning_effort` is rejected, confirming `--no-reasoning-effort` is the right call.

In [ ]:
# --- endpoint health -----------------------------------------------------------------
import json, time
import openai

client = openai.OpenAI(base_url=BASE_URL, api_key=os.environ[API_KEY_ENV])

served = [m.id for m in client.models.list().data]
print("served models:", served)
assert MODEL in served, f"{MODEL!r} not served; vLLM is offering {served}"

# The exact schema the harness sends, so this proves the real request shape.
t0 = time.time()
smoke = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are an expert cryptanalyst working on classical ciphers."},
        {"role": "user", "content": "Ciphertext (11 characters): WKHTXLFNIRA\nIt is a Caesar shift. Decrypt it."},
    ],
    max_tokens=512,
    response_format={
        "type": "json_schema",
        "json_schema": {"name": "answer", "strict": True, "schema": paradigms.ANSWER_SCHEMA},
    },
)
elapsed = time.time() - t0

payload = json.loads(smoke.choices[0].message.content)   # raises if the schema was ignored
print(f"\nstructured output OK  ({elapsed:.1f}s, "
      f"{smoke.usage.completion_tokens} out tok, "
      f"{smoke.usage.completion_tokens / elapsed:.0f} tok/s)")
print(json.dumps(payload, indent=2)[:600])

In [ ]:
# --- confirm reasoning_effort is rejected --------------------------------------------
# If this endpoint ever *accepts* the field, drop --no-reasoning-effort below so the
# results file records the level actually sent instead of "unset".
try:
    client.chat.completions.create(
        model=MODEL, messages=[{"role": "user", "content": "hi"}],
        max_tokens=8, reasoning_effort="high",
    )
    REASONING_EFFORT = True
    print("accepted -- remove --no-reasoning-effort from the run below")
except Exception as exc:
    REASONING_EFFORT = False
    print(f"rejected as expected ({type(exc).__name__}) -- keeping --no-reasoning-effort")

---
## 3. The data, and what the model actually sees

Before spending the run, confirm the instance counts and look at one fully-built prompt.
The leak rule is enforced in `kryptos.eval.tiers`, not here, but seeing a rendered prompt
is the cheap check that `answer`, `answer_readable` and `solution` are absent.

In [ ]:
# --- dataset ------------------------------------------------------------------------
from collections import Counter
from datasets import load_dataset

rows_by_config = {}
for cfg in CONFIGS:
    rows = [dict(r) for r in load_dataset("sartajbhuvaji/kryptos-bench", cfg, split="test")]
    rows_by_config[cfg] = rows
    tier_mix = Counter(tiers.default_tier(r) for r in rows)
    mix = "  ".join(f"tier {t}: {n}" for t, n in sorted(tier_mix.items()))
    print(f"{cfg:<26} {len(rows):>3} rows   {mix}")

TOTAL = sum(len(r) for r in rows_by_config.values())
print(f"\n{TOTAL} instances total")

In [ ]:
# --- one rendered prompt, exactly as the model receives it ---------------------------
row  = rows_by_config["isomorph_quagmire"][0]
tier = tiers.default_tier(row)

print(f"--- SYSTEM (tier {tier}) " + "-" * 50)
print(tiers.system_prompt(tier))
print(f"\n--- USER " + "-" * 62)
print(tiers.build_prompt(row, tier))

# The check that matters: no ground truth anywhere in the prompt.
rendered = tiers.system_prompt(tier) + tiers.build_prompt(row, tier)
leaked = [f for f in tiers.FORBIDDEN_FIELDS
          if row.get(f) and str(row[f])[:40] in rendered]
print("\n" + "-" * 70)
print("leaked ground-truth fields:", leaked or "none")
assert not leaked

---
## 4. Concurrency probe

`--concurrency` is how many instances the harness keeps in flight. vLLM batches whatever
arrives into a single forward pass, so raising it converts idle GPU into throughput at
almost no cost — until the KV cache is exhausted, at which point vLLM starts preempting
and recomputing sequences and throughput goes *down*.

That knee depends on the actual prompt and output lengths, so measure it rather than
guessing. This sends the same small set of real benchmark prompts at several concurrency
levels with a short output cap, and reports aggregate output tokens/sec.

It costs a couple of minutes of GPU time and can save a lot on a 204-instance run. Skip
it by setting `PROBE = False`.

In [ ]:
# --- concurrency probe ---------------------------------------------------------------
PROBE        = True
PROBE_LEVELS = [1, 4, 8, 16, 24, 32]
PROBE_N      = 16      # prompts per level
PROBE_TOKENS = 512     # short cap -- measuring scaling, not solution quality

from concurrent.futures import ThreadPoolExecutor

def _ask(prompt_pair):
    system, user = prompt_pair
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "system", "content": system}, {"role": "user", "content": user}],
        max_tokens=PROBE_TOKENS,
        response_format={"type": "json_schema",
                         "json_schema": {"name": "answer", "strict": True,
                                         "schema": paradigms.ANSWER_SCHEMA}},
    )
    return r.usage.completion_tokens

if PROBE:
    sample = rows_by_config["isomorph_quagmire"][:PROBE_N]
    prompts = [(tiers.system_prompt(tiers.default_tier(r)),
                tiers.build_prompt(r, tiers.default_tier(r))) for r in sample]

    print(f"{'concurrency':>12}{'wall s':>10}{'out tok':>10}{'tok/s':>10}{'speedup':>10}")
    baseline_rate, best = None, (0, None)
    for level in PROBE_LEVELS:
        t0 = time.time()
        with ThreadPoolExecutor(max_workers=level) as pool:
            toks = sum(pool.map(_ask, prompts))
        dt = time.time() - t0
        rate = toks / dt
        baseline_rate = baseline_rate or rate
        if rate > best[0]:
            best = (rate, level)
        print(f"{level:>12}{dt:>10.1f}{toks:>10}{rate:>10.0f}{rate/baseline_rate:>9.1f}x")

    CONCURRENCY = best[1]
    print(f"\nfastest at concurrency {CONCURRENCY} ({best[0]:.0f} tok/s); using it for the run")
    print("if the curve is still rising at the last level, extend PROBE_LEVELS")

---
## 5. Run

One invocation per config. Each writes JSONL into `GLM-4.6-Flash-text/results/` as it
goes, so an interrupted run loses nothing.

`--resume` skips instances already answered in the output file and **retries refusals and
errors**, which makes this cell safe to re-run — a partial run continues rather than
duplicating work or double-counting.

This is the expensive cell. Expect it to take a while: 204 instances, up to 8192 output
tokens each, 8 in flight.

In [ ]:
# --- run ----------------------------------------------------------------------------
import time

run_started = time.time()
timings = {}

for cfg in CONFIGS:
    out = RESULTS / f"{cfg}.jsonl"
    argv = [
        "--config", cfg,
        "--model", MODEL,
        "--provider", "openai",
        "--base-url", BASE_URL,
        "--api-key-env", API_KEY_ENV,
        "--paradigm", "cot",
        "--concurrency", str(CONCURRENCY),
        "--provider-param", f"max_tokens={MAX_TOKENS}",
        "--out", str(out),
        "--resume",
    ]
    if not REASONING_EFFORT:
        argv.append("--no-reasoning-effort")

    print("=" * 78)
    print(f"  {cfg}")
    print("=" * 78)
    t0 = time.time()
    rc = run_benchmark.main(argv)
    timings[cfg] = time.time() - t0
    print(f"\n[{cfg}] exit {rc} in {timings[cfg]/60:.1f} min\n")

WALL_CLOCK = time.time() - run_started
print(f"total wall clock: {WALL_CLOCK/60:.1f} min")

---
## 6. Results

`kryptos.eval.report` is the same aggregation the CLI uses, so these numbers are the ones
`python -m kryptos.eval.report` would print from the committed files.

In [ ]:
# --- load and summarise --------------------------------------------------------------
paths = sorted(RESULTS.glob("*.jsonl"))
records = reporting.load(paths)
records, duplicates = reporting.deduplicate(records)

summary = reporting.summarise(records)
print(f"loaded {len(records)} records from {len(paths)} files"
      + (f"  ({duplicates} duplicate(s) dropped)" if duplicates else ""))
print()
print(summary)

In [ ]:
# --- per-config breakdown ------------------------------------------------------------
print(reporting.render_breakdown(records, "config"))

In [ ]:
# --- the headline: memorisation vs cryptanalysis -------------------------------------
# `baseline` is K1-K4, which are published; the isomorph configs are freshly generated.
# A gap here is the benchmark's reason to exist. report.headline() is the harness's own
# comparison -- deliberately unpaired, because the two sides are different instances by
# design and pairing would mean pretending K1 corresponds to some particular synthetic.
h = reporting.headline(records, MODEL)

def fmt(cer, ci):
    if cer is None:
        return "n/a"
    return f"{cer:.4f}" + (f" +/- {ci:.4f}" if ci is not None else "")

print(f"{'baseline (published)':<24} mean CER {fmt(h.left_cer, h.left_ci95):<20}"
      f" exact solves {h.left_solved}")
print(f"{'isomorph (generated)':<24} mean CER {fmt(h.right_cer, h.right_ci95):<20}"
      f" exact solves {h.right_solved}")
print()
if h.cer_gap is None:
    print("no gap computable -- one side produced nothing scoreable")
else:
    print(f"gap (baseline - isomorph): {h.cer_gap:+.4f}")
    print("negative => baseline scored better, i.e. recall the generated instances deny")

### Charts

Two views. The strip plot carries the whole distribution rather than only its mean — at
these sample sizes a mean CER hides whether the model is uniformly bad or bimodal
(solving some instances outright and failing others), and those are very different
findings. The outcome mix separates *wrong answers* from *no answer*, so a harness
failure never gets read as a cryptanalysis failure.

In [ ]:
# --- chart setup ---------------------------------------------------------------------
import matplotlib.pyplot as plt
import numpy as np

SURFACE   = "#fcfcfb"
INK       = "#0b0b0b"
INK_SOFT  = "#52514e"
GRID      = "#e3e2df"
BLUE      = "#2a78d6"   # categorical slot 1
AQUA      = "#1baf7a"   # slot 3
YELLOW    = "#eda100"   # slot 4
VIOLET    = "#4a3aa7"   # slot 7
RED       = "#e34948"   # slot 8

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE,
    "text.color": INK, "axes.labelcolor": INK_SOFT,
    "xtick.color": INK_SOFT, "ytick.color": INK_SOFT,
    "axes.edgecolor": GRID, "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
})

SHORT = {"baseline": "baseline", "isomorph_quagmire": "quagmire",
         "isomorph_transposition": "transposition", "isomorph_composite": "composite",
         "isomorph_nulls": "nulls"}
ORDER = ["baseline", "isomorph_quagmire", "isomorph_transposition",
         "isomorph_composite", "isomorph_nulls"]

def group_of(rec):
    """Which config bucket a record belongs to.

    Baseline rows carry their passage (K1..K4) in `config` rather than the config name,
    so report.family() is what folds those four back together -- the same rule the
    harness's own headline comparison uses.
    """
    return "baseline" if reporting.family(rec) == "baseline" else str(rec["config"])

In [ ]:
# --- chart 1: CER distribution per config --------------------------------------------
# One series, so no legend -- the title names it. Jitter is deterministic so the figure
# is reproducible across re-runs of this cell.
groups = {cfg: [r["cer"] for r in records
                if group_of(r) == cfg and r.get("cer") is not None]
          for cfg in ORDER}
groups = {k: v for k, v in groups.items() if v}

fig, ax = plt.subplots(figsize=(9, 0.85 * len(groups) + 1.6))
rng = np.random.default_rng(0)

for i, (cfg, vals) in enumerate(groups.items()):
    y = i + rng.uniform(-0.16, 0.16, len(vals))
    ax.scatter(vals, y, s=34, color=BLUE, alpha=0.45,
               edgecolors=SURFACE, linewidths=1.2, zorder=3)   # 2px surface ring
    m = float(np.mean(vals))
    ax.scatter([m], [i], marker="|", s=520, color=INK, linewidths=2.4, zorder=4)
    ax.annotate(f"mean {m:.3f}   n={len(vals)}", (m, i), xytext=(0, 15),
                textcoords="offset points", ha="center", fontsize=9, color=INK_SOFT)

ax.set_yticks(range(len(groups)), [SHORT[c] for c in groups])
ax.set_xlabel("character error rate  (0 = exact, lower is better)")
ax.set_xlim(-0.03, max(1.05, max(max(v) for v in groups.values()) * 1.05))
ax.set_ylim(-0.6, len(groups) - 0.25)
ax.invert_yaxis()
ax.xaxis.grid(True, color=GRID, linewidth=1)
ax.set_axisbelow(True)
ax.set_title(f"CER by config — {MODEL}", loc="left", fontsize=12, color=INK, pad=26)

fig.tight_layout()
fig.savefig(RESULTS / "cer_by_config.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
# --- chart 2: outcome mix per config -------------------------------------------------
# Separates wrong answers from no answer. A refusal or a transport error is a harness or
# policy event, not a cryptanalysis result, and scoring it as CER 1.0 would fold the two
# together -- so they get their own bands here.
def outcome(rec):
    if rec.get("error"):   return "error"
    if rec.get("refused"): return "refused"
    cer = rec.get("cer")
    if cer is None:        return "unscored (K4)"
    if cer == 0.0:         return "exact"
    if cer <= 0.10:        return "near (CER ≤ 0.10)"
    return "wrong"

# Band order and hues are fixed, never cycled. Slot 2 (orange) is deliberately NOT used
# for `refused`: beside slot 8 (red) it fails both colourblind-separation gates and the
# normal-vision floor (CVD dE 5.6 against a 6.0 floor, normal dE 7.1 against 15.0), so
# the two "no answer" bands would be indistinguishable. Violet clears both (22.7 / 33.6).
# Aqua and yellow sit under 3:1 against this surface, which is why the in-bar labels and
# the printed table below are load-bearing rather than decorative.
BANDS  = ["exact", "near (CER ≤ 0.10)", "wrong", "unscored (K4)", "refused", "error"]
COLORS = [AQUA, BLUE, YELLOW, INK_SOFT, VIOLET, RED]

counts = {cfg: Counter(outcome(r) for r in records if group_of(r) == cfg)
          for cfg in ORDER}
counts = {k: v for k, v in counts.items() if sum(v.values())}

fig, ax = plt.subplots(figsize=(9, 0.62 * len(counts) + 2.1))
ys = np.arange(len(counts))
left = np.zeros(len(counts))

for band, colour in zip(BANDS, COLORS):
    widths = np.array([counts[c].get(band, 0) for c in counts], dtype=float)
    if not widths.any():
        continue
    ax.barh(ys, widths, left=left, height=0.6, color=colour,
            edgecolor=SURFACE, linewidth=2, label=band, zorder=3)   # 2px surface gap
    for y, w, l in zip(ys, widths, left):
        if w >= 4:                                   # direct-label only where it fits
            ax.text(l + w / 2, y, int(w), ha="center", va="center",
                    fontsize=9, color=SURFACE, fontweight="bold", zorder=4)
    left += widths

ax.set_yticks(ys, [SHORT[c] for c in counts])
ax.set_xlabel("instances")
ax.invert_yaxis()
ax.xaxis.grid(True, color=GRID, linewidth=1)
ax.set_axisbelow(True)
ax.set_title(f"Outcome mix — {MODEL}", loc="left", fontsize=12, color=INK, pad=10)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=3,
          frameon=False, fontsize=9)

fig.tight_layout()
fig.savefig(RESULTS / "outcome_mix.png", dpi=160, bbox_inches="tight")
plt.show()

# The table view, so identity is never carried by colour alone.
print(f"{'config':<16}" + "".join(f"{b:>20}" for b in BANDS))
for cfg, c in counts.items():
    print(f"{SHORT[cfg]:<16}" + "".join(f"{c.get(b, 0):>20}" for b in BANDS))

In [ ]:
# --- failure audit -------------------------------------------------------------------
# Anything here is a harness, endpoint or policy problem rather than a model score.
# `unparsed_response` in particular means structured output silently stopped working.
failed = [r for r in records if r.get("error") or r.get("refused")]
if not failed:
    print("no refusals, no errors -- every instance produced a scoreable answer")
else:
    print(f"{len(failed)} of {len(records)} attempts produced no scoreable answer:\n")
    for label, n in Counter(r.get("error") or f"refused:{r.get('refusal_category')}"
                            for r in failed).most_common():
        print(f"  {n:>4}  {label}")
    print("\nre-running the run cell retries these -- --resume skips only successes")

---
## 7. Provenance and cost

A score without the machine it ran on is an anecdote. This writes the environment block,
the run's timings and a derived cost next to the results.

The cost figure is **amortised GPU time**, not a list price: a self-hosted model has no
per-token rate, so the honest number is what the A100 hour actually bought. That is why
the run is left unpriced in the harness (`--price` is not passed) rather than given an
invented per-million rate that would then flow into cross-model cost comparisons as if it
were a real tariff.

In [ ]:
# --- provenance ----------------------------------------------------------------------
import datetime, subprocess

A100_USD_PER_HOUR = 1.99          # Lambda gpu_1x_a100_sxm4, on-demand

# Captured on the instance with: PYTHON=~/venvs/vllm/bin/python bash tools/gpu_env.sh --json
# Paste that JSON here (or point GPU_ENV_FILE at a copy) so the block travels with the run.
GPU_ENV_FILE = RESULTS / "gpu_env.json"
gpu_env = json.loads(GPU_ENV_FILE.read_text()) if GPU_ENV_FILE.exists() else {
    "note": "not captured -- run tools/gpu_env.sh --json on the instance and save it here"
}

in_tok  = sum(r.get("input_tokens", 0) for r in records)
out_tok = sum(r.get("output_tokens", 0) for r in records)
gpu_usd = A100_USD_PER_HOUR * WALL_CLOCK / 3600

meta = {
    "model": HF_MODEL,
    "served_as": MODEL,
    "dataset": "sartajbhuvaji/kryptos-bench",
    "configs": CONFIGS,
    "paradigm": "cot",
    "delimited": False,
    "reasoning_effort_sent": REASONING_EFFORT,
    "max_tokens": MAX_TOKENS,
    "concurrency": CONCURRENCY,
    "instances": len(records),
    "wall_clock_seconds": round(WALL_CLOCK, 1),
    "per_config_seconds": {k: round(v, 1) for k, v in timings.items()},
    "input_tokens": in_tok,
    "output_tokens": out_tok,
    "throughput_output_tok_per_s": round(out_tok / WALL_CLOCK, 1) if WALL_CLOCK else None,
    "gpu_cost_usd": round(gpu_usd, 2),
    "gpu_usd_per_hour": A100_USD_PER_HOUR,
    "amortised_usd_per_million_output_tokens": (
        round(gpu_usd / (out_tok / 1e6), 2) if out_tok else None),
    "results_version": results_mod.RESULTS_VERSION,
    "run_finished_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "environment": gpu_env,
}

(RESULTS / "run_metadata.json").write_text(json.dumps(meta, indent=2) + "\n")
print(json.dumps(meta, indent=2))

In [ ]:
# --- SUMMARY.md, committed next to the results ---------------------------------------
lines = [
    f"# Kryptos-bench — `{HF_MODEL}`", "",
    f"- **Run:** {meta['run_finished_utc']}",
    "- **Serving:** vLLM on Lambda `gpu_1x_a100_sxm4`, BF16, 32768-token context",
    "- **Axes:** paradigm `cot`, raw ciphertext, effort `unset`, default tier per row",
    f"- **Instances:** {summary.instances}  "
    f"({summary.scored} scored, {summary.frontier} frontier, "
    f"{summary.refused} refused, {summary.errored} errored)",
    f"- **Wall clock:** {WALL_CLOCK/60:.1f} min  (${meta['gpu_cost_usd']} of A100 time)",
    f"- **Throughput:** {meta['throughput_output_tok_per_s']} output tok/s aggregate",
    "",
    "## Per-config", "",
    "```", reporting.render_breakdown(records, "config"), "```", "",
    "## Per-tier", "",
    "```", reporting.render_breakdown(records, "tier"), "```", "",
    "## Memorisation vs cryptanalysis", "",
    "`baseline` is the published K1–K4. The `isomorph_*` configs are freshly generated",
    "with novel keys and novel plaintext, so they cannot have been in training data.",
    "A low CER on the first with a high CER on the second is recall, not cryptanalysis.",
    "",
    f"- baseline mean CER: {fmt(h.left_cer, h.left_ci95)}  ({h.left_solved} exact)",
    f"- isomorph mean CER: {fmt(h.right_cer, h.right_ci95)}  ({h.right_solved} exact)",
    f"- gap: {'n/a' if h.cer_gap is None else format(h.cer_gap, '+.4f')}",
    "",
    "## Caveats", "",
    "- Tier pass marks are asserted by the design document, **not calibrated** against an",
    "  observed distribution. Treat `passed` as provisional.",
    "- `tool_use` was not run: it needs a server-side sandbox the OpenAI wire format has",
    "  no equivalent for, so this is chain-of-thought only and is not comparable to a",
    "  Claude `tool_use` arm.",
    "- Cost is amortised GPU time, not a per-token tariff.",
    "",
    "![CER by config](cer_by_config.png)", "",
    "![Outcome mix](outcome_mix.png)", "",
]
(RESULTS / "SUMMARY.md").write_text("\n".join(lines), encoding="utf-8")
print("\n".join(lines))

---
## 8. Teardown

The A100 bills at **$1.99/hr** for as long as it exists. When the results above look
right, terminate it:

```bash
cd ../../..//unfuse            # wherever the unfuse checkout is
bash tools/launch_lambda.sh terminate <INSTANCE_ID>
```

Then verify it is actually gone — a terminate that silently failed still bills:

```bash
curl -sS -u "$LAMBDA_API_KEY:" https://cloud.lambda.ai/api/v1/instances
```

An empty `data` array means nothing is running.

Everything needed to re-analyse this run without paying again is already on disk in
`GLM-4.6-Flash-text/results/` — the per-instance JSONL, the metadata block and the
summary. Re-running the notebook against a fresh instance with `--resume` will skip
what is already answered.